# Machine Learning for Spatial Data — A0 + A
## Foundations, and why standard cross-validation *leaks* on spatial data

**Companion to deck `09-spatial-machine-learning` (Module A — Foundations & the spatial catch: leakage and spatial cross-validation).**

The course so far modelled space to *explain* (Moran's I, GWR, kriging, spatial
econometrics). Here we switch goal to *predict*, and meet machine learning as a
first-class predictor. We reproduce, on San Diego Airbnb, the effect Ploton et al.
(2020, *Nature Communications*) showed for forest biomass: a random-CV R² collapses
under spatial CV.

This notebook shows, step by step, how to:
1. Load the **San Diego Airbnb** listings (`data/airbnb/airbnb_clean.csv`) with `pandas`.
2. Assemble features `X` from listing attributes **plus raw coordinates** (`longitude`, `latitude`) and a target `y` = **log price** via `np.log1p` — the naive but common way to "let the model use location".
3. Fit a first **Random Forest** (`RandomForestRegressor`) on a random `train_test_split`, and contrast **train R²** with **test R²** to expose **over-fitting**.
4. Diagnose **spatial leakage**: under Tobler's law a random split scatters neighbours across train and test, so the model interpolates between near-duplicates.
5. Build **spatial folds** by clustering the coordinates with **k-means** (`KMeans` on lon/lat) and holding out whole clusters via `GroupKFold` — the clustering-based spatial CV of Sun, Hu et al. (2023, *Handbook of GeoAI*).
6. Compare **random `KFold` CV** against **spatial `GroupKFold` CV** with `cross_val_score`, and quantify the **optimism** (random − spatial) in R² points.
7. Weigh the **debate**: Wadoux et al. (2021, *Ecological Modelling*) argue spatial CV can be *over*-pessimistic for map accuracy — report both scores and state the prediction task.

> Dataset: `data/airbnb/airbnb_clean.csv` — San Diego Airbnb listings, the same regression case used in Rey, Arribas-Bel & Wolf, *Geographic Data Science with Python*.
> Source: https://geographicdata.science/book/


In [1]:
import warnings
warnings.filterwarnings("ignore")  # quiet harmless numpy2/BLAS matmul notices
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import KFold, GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score

RNG = 42
df = pd.read_csv("data/airbnb/airbnb_clean.csv")
print(df.shape)
df.head()

(3173, 11)


,price,accommodates,bedrooms,beds,bathrooms,guests_included,reviews_per_month,number_of_reviews,host_listings_count,longitude,latitude
0,85.0,4,1.0,1.0,1.0,2,4.07,207,3.0,-9.180527e+06,5.302872e+06
1,150.0,4,1.0,1.0,1.0,1,1.48,43,6.0,-9.180125e+06,5.303187e+06
2,975.0,11,5.0,7.0,4.5,10,1.15,20,2.0,-9.180411e+06,5.302141e+06
3,450.0,6,3.0,3.0,2.0,6,0.89,38,2.0,-9.180635e+06,5.302496e+06
4,120.0,2,1.0,1.0,1.0,1,2.45,17,1.0,-9.180163e+06,5.302230e+06


## A0 — A first machine-learning model

Target `y` = log price (log stabilises the skew). Features `X` = listing
attributes **plus the coordinates** (longitude, latitude) — a naive but common
way to "let the model use location". `fit` on a training slice, `predict` on a
held-out test slice.

In [2]:
feat = ["accommodates", "bedrooms", "beds", "bathrooms", "guests_included",
        "reviews_per_month", "number_of_reviews", "host_listings_count",
        "longitude", "latitude"]
X = df[feat].fillna(0.0).values
y = np.log1p(df["price"].values)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RNG)
rf = RandomForestRegressor(n_estimators=300, random_state=RNG, n_jobs=2)
rf.fit(X_tr, y_tr)

r2_train = r2_score(y_tr, rf.predict(X_tr))
r2_test = r2_score(y_te, rf.predict(X_te))
print(f"train R^2 = {r2_train:.3f}")
print(f"test  R^2 = {r2_test:.3f}   (random split)")

train R^2 = 0.955
test  R^2 = 0.667   (random split)


The gap between train R² (near 1) and test R² is **over-fitting**: a Random
Forest memorises the training rows. The test split gives an *honest-looking*
score — but on spatial data it is still too optimistic. Why? Next.

## A — The spatial catch: leakage

Tobler's law: near things are similar. A **random** train/test split scatters
neighbours across both sets, so for almost every test listing there is a very
similar training listing next door. The model "peeks" — it interpolates between
near-duplicates instead of generalising to *new places*.

**Diagnosis:** compare a random k-fold CV against a **spatial** k-fold CV, where
whole geographic clusters are held out. We build spatial folds by clustering the
coordinates (k-means on lon/lat) and using `GroupKFold` — the clustering-based
spatial CV of Sun, Hu et al. (2023, *Handbook of GeoAI*).

In [3]:
K = 10
rf_cv = RandomForestRegressor(n_estimators=300, random_state=RNG, n_jobs=2)

# random k-fold
kf = KFold(n_splits=K, shuffle=True, random_state=RNG)
r2_random = cross_val_score(rf_cv, X, y, cv=kf, scoring="r2", n_jobs=2)

# spatial k-fold: hold out whole geographic clusters
groups = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit_predict(
    df[["longitude", "latitude"]].values)
gkf = GroupKFold(n_splits=K)
r2_spatial = cross_val_score(rf_cv, X, y, groups=groups, cv=gkf,
                             scoring="r2", n_jobs=2)

print(f"random  k-fold R^2 = {r2_random.mean():.3f}  +/- {r2_random.std():.3f}")
print(f"spatial k-fold R^2 = {r2_spatial.mean():.3f}  +/- {r2_spatial.std():.3f}")
print(f"optimism (random - spatial) = "
      f"{r2_random.mean() - r2_spatial.mean():.3f} R^2 points")

random  k-fold R^2 = 0.664  +/- 0.043
spatial k-fold R^2 = 0.539  +/- 0.131
optimism (random - spatial) = 0.125 R^2 points


The random score is **higher** than the spatial score. That gap is the leakage:
it is performance you will *not* get when predicting a neighbourhood the model
has never seen. This is the same failure mode Ploton et al. (2020) quantified for
forest biomass, where random 10-fold R² = 0.53 collapsed to 0.14 under spatial
44-fold CV — near a null model.

**Caveat (teach the debate):** Wadoux et al. (2021, *Ecological Modelling*) argue
spatial CV can be *over*-pessimistic for map-accuracy assessment. The honest
answer depends on the prediction task: interpolating within sampled areas vs.
extrapolating to new regions. Report *both* numbers and say which question you
are answering.

**Lesson —**
- ML gives you `fit`/`predict` and strong non-linear fits — but the score is only
  as honest as the split.
- On spatial data, **validate spatially**. The gap above is the price of Tobler's
  law. Every later module (RFsp, GNNs, location encoders) is validated this way.